#### 1. Loading MITRE ATT&CK data
The Enterprise ATT&CK dataset is loaded to examine its structure and identify techniques, mitigations, and their relationships.

In [1]:
import json
import pandas as pd
from pathlib import Path

data_dir = Path("../data")

if not data_dir.exists():
    data_dir = Path("data")

with open(data_dir / "raw/enterprise-attack.json",
          "r", encoding="utf-8") as f:
    mitre = json.load(f)

print("Total objects:", len(mitre["objects"]))

Total objects: 26699


#### 2. Examining the MITRE data structure
The different object types are counted to identify the information relevant to the project.

In [2]:
objects = pd.DataFrame(mitre["objects"])

print("Object types:")
display(objects["type"].value_counts())

Object types:


type
relationship                  21822
x-mitre-analytic               1773
attack-pattern                  867
malware                         750
x-mitre-detection-strategy      708
course-of-action                268
intrusion-set                   193
x-mitre-data-component          109
tool                             95
campaign                         57
x-mitre-data-source              38
x-mitre-tactic                   15
x-mitre-collection                1
x-mitre-matrix                    1
identity                          1
marking-definition                1
Name: count, dtype: int64

#### 3. Extracting attack techniques
Active attack techniques are extracted with their MITRE identifiers, names, descriptions, and tactics.

In [3]:
# selects active attack techniques
techniques = objects[
    (objects["type"] == "attack-pattern")
    & (objects["revoked"] != True)
    & (objects["x_mitre_deprecated"] != True)
].copy()

# extracts the MITRE technique ID (e.g. T1566)
def get_mitre_id(refs):
    if not isinstance(refs, list):
        return None

    for ref in refs:
        if ref.get("source_name") == "mitre-attack":
            return ref.get("external_id")

    return None

techniques["technique_id"] = techniques["external_references"].apply(get_mitre_id)

# extracts tactics from the kill chain information
def get_tactics(phases):
    if not isinstance(phases, list):
        return []

    return [
        phase["phase_name"]
        for phase in phases
        if phase.get("kill_chain_name") == "mitre-attack"
    ]

techniques["tactics"] = techniques["kill_chain_phases"].apply(get_tactics)

# keeps relevant columns
techniques = techniques[
    ["id", "technique_id", "name", "description", "tactics"]
].reset_index(drop=True)

print("Active techniques:", len(techniques))
display(techniques.head())

Active techniques: 706


,id,technique_id,name,description,tactics
0,attack-pattern--0042a9f5-f053-4769-b3ef-9ad018...,T1055.011,Extra Window Memory Injection,Adversaries may inject malicious code into a p...,"[stealth, privilege-escalation]"
1,attack-pattern--005a06c6-14bf-4118-afa0-ebcd8a...,T1053.005,Scheduled Task,Adversaries may abuse the Windows Task Schedul...,"[execution, persistence, privilege-escalation]"
2,attack-pattern--005cc321-08ce-4d17-b1ea-cb5275...,T1205.002,Socket Filters,Adversaries may attach filters to a network so...,"[stealth, persistence, command-and-control]"
3,attack-pattern--00f90846-cbd1-4fc5-9233-df5c2b...,T1560.001,Archive via Utility,Adversaries may use utilities to compress and/...,[collection]
4,attack-pattern--01327cde-66c4-4123-bf34-5f258d...,T1021.005,VNC,Adversaries may use [Valid Accounts](https://a...,[lateral-movement]


In [5]:
print("Missing technique IDs:", techniques["technique_id"].isna().sum())
print("Duplicate technique IDs:", techniques["technique_id"].duplicated().sum())

display(techniques[techniques["technique_id"] == "T1566"])

Missing technique IDs: 0
Duplicate technique IDs: 0


,id,technique_id,name,description,tactics
446,attack-pattern--a62a8db3-f23a-4d8f-afd6-9dbc77...,T1566,Phishing,Adversaries may send phishing messages to gain...,[initial-access]


#### 4. Extracting mitigations
Active MITRE mitigations are extracted with their identifiers, names, and descriptions.

In [6]:
# selects active mitigations
mitigations = objects[
    (objects["type"] == "course-of-action")
    & (objects["revoked"] != True)
    & (objects["x_mitre_deprecated"] != True)
].copy()

# extracts MITRE mitigation IDs (e.g. M1031)
mitigations["mitigation_id"] = mitigations["external_references"].apply(
    get_mitre_id
)

# keeps relevant columns
mitigations = mitigations[
    ["id", "mitigation_id", "name", "description"]
].reset_index(drop=True)

print("Active mitigations:", len(mitigations))
display(mitigations.head())

Active mitigations: 44


,id,mitigation_id,name,description
0,course-of-action--12241367-a8b7-49b4-b86e-2236...,M1031,Network Intrusion Prevention,Use intrusion detection signatures to block tr...
1,course-of-action--15437c6d-b998-4a36-be41-4ace...,M1016,Vulnerability Scanning,Vulnerability scanning involves the automated ...
2,course-of-action--1dcaeb21-9348-42ea-950a-f842...,M1035,Limit Access to Resource Over Network,"Restrict access to network resources, such as ..."
3,course-of-action--20a2baeb-98c2-4901-bad7-dc62...,M1029,Remote Data Storage,Remote Data Storage focuses on moving critical...
4,course-of-action--20f6a9df-37c4-4e20-9e47-0259...,M1037,Filter Network Traffic,Employ network appliances and endpoint softwar...


#### 5. Extracting technique-mitigation relationships
MITRE relationships are used to identify which defensive measures mitigate specific attack techniques.

In [7]:
# selects mitigation relationships
relationships = objects[
    (objects["type"] == "relationship")
    & (objects["relationship_type"] == "mitigates")
    & (objects["revoked"] != True)
    & (objects["x_mitre_deprecated"] != True)
].copy()

# keeps relationships between active mitigations and techniques
relationships = relationships[
    relationships["source_ref"].isin(mitigations["id"])
    & relationships["target_ref"].isin(techniques["id"])
]

# keeps relevant columns
relationships = relationships[
    ["source_ref", "target_ref"]
].drop_duplicates().reset_index(drop=True)

print("Technique-mitigation relationships:", len(relationships))
display(relationships.head())

Technique-mitigation relationships: 1477


,source_ref,target_ref
0,course-of-action--21da4fd4-27ad-4e9c-b93d-0b9b...,attack-pattern--43c9bc06-715b-42db-972f-52d25c...
1,course-of-action--78bb71be-92b4-46de-acd6-5f99...,attack-pattern--67073dde-d720-45ae-83da-b12d5e...
2,course-of-action--20f6a9df-37c4-4e20-9e47-0259...,attack-pattern--8868cb5b-d575-4a60-acb2-07d373...
3,course-of-action--b5dbb4c5-b0b1-40b1-80b6-e9e8...,attack-pattern--2b742742-28c3-4e1b-bab7-8350d6...
4,course-of-action--9bb9e696-bff8-4ae1-9454-961f...,attack-pattern--005a06c6-14bf-4118-afa0-ebcd8a...


#### 6. Connecting techniques and mitigations
The MITRE relationships are combined with technique and mitigation information to create a readable dataset.

In [8]:
# adds mitigation information
technique_mitigations = relationships.merge(
    mitigations[["id", "mitigation_id", "name"]],
    left_on="source_ref",
    right_on="id",
    how="left",
    validate="many_to_one"
)

technique_mitigations = technique_mitigations.rename(
    columns={"name": "mitigation_name"}
).drop(columns="id")

# adds technique information
technique_mitigations = technique_mitigations.merge(
    techniques[["id", "technique_id", "name"]],
    left_on="target_ref",
    right_on="id",
    how="left",
    validate="many_to_one"
)

technique_mitigations = technique_mitigations.rename(
    columns={"name": "technique_name"}
).drop(columns="id")

# keeps readable columns
technique_mitigations = technique_mitigations[
    ["technique_id", "technique_name",
     "mitigation_id", "mitigation_name"]
].drop_duplicates().reset_index(drop=True)

print("Connected technique-mitigation pairs:", len(technique_mitigations))

display(technique_mitigations.head())

Connected technique-mitigation pairs: 1477


,technique_id,technique_name,mitigation_id,mitigation_name
0,T1659,Content Injection,M1021,Restrict Web-Based Content
1,T1595,Active Scanning,M1056,Pre-compromise
2,T1205.001,Port Knocking,M1037,Filter Network Traffic
3,T1566.002,Spearphishing Link,M1054,Software Configuration
4,T1053.005,Scheduled Task,M1026,Privileged Account Management


#### 7. Validating MITRE datasets
The extracted identifiers and technique-mitigation relationships are checked for missing values and duplicates.

In [9]:
print("Techniques:", len(techniques))
print("Mitigations:", len(mitigations))
print("Relationships:", len(technique_mitigations))

print("\nMissing technique IDs:", techniques["technique_id"].isna().sum())
print("Missing mitigation IDs:", mitigations["mitigation_id"].isna().sum())

print("Duplicate technique IDs:", techniques["technique_id"].duplicated().sum())
print("Duplicate mitigation IDs:", mitigations["mitigation_id"].duplicated().sum())

print("\nMissing relationship values:")
print(technique_mitigations.isna().sum())

# example: MITRE mitigations for phishing
display(
    technique_mitigations[
        technique_mitigations["technique_id"] == "T1566"
    ]
)

Techniques: 706
Mitigations: 44
Relationships: 1477

Missing technique IDs: 0
Missing mitigation IDs: 0
Duplicate technique IDs: 0
Duplicate mitigation IDs: 0

Missing relationship values:
technique_id       0
technique_name     0
mitigation_id      0
mitigation_name    0
dtype: int64


,technique_id,technique_name,mitigation_id,mitigation_name
98,T1566,Phishing,M1047,Audit
651,T1566,Phishing,M1031,Network Intrusion Prevention
681,T1566,Phishing,M1054,Software Configuration
936,T1566,Phishing,M1021,Restrict Web-Based Content
1353,T1566,Phishing,M1049,Antivirus/Antimalware
1365,T1566,Phishing,M1017,User Training


#### 8. Saving MITRE datasets
The extracted techniques, mitigations, and their documented relationships are saved for integration with the CISA exercises.

In [10]:
processed_dir = data_dir / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

techniques.to_csv(
    processed_dir / "mitre_techniques.csv",
    index=False,
    encoding="utf-8-sig"
)

mitigations.to_csv(
    processed_dir / "mitre_mitigations.csv",
    index=False,
    encoding="utf-8-sig"
)

technique_mitigations.to_csv(
    processed_dir / "technique_mitigations.csv",
    index=False,
    encoding="utf-8-sig"
)

print("MITRE datasets saved successfully.")

MITRE datasets saved successfully.
